# LangChain 과제 1-1 ~ 1-3
공통 준비 셀을 먼저 실행하세요. `.env`에 `OPENAI_API_KEY=...`를 설정해야 합니다.

In [23]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv(override=True)

llm = ChatOpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
    model="openai/gpt-oss-120b",
    temperature=0
)


## 1-1 AI 요리사

In [24]:
# 1. 프롬프트 생성
prompt = PromptTemplate.from_template("""
당신은 AI 요리사입니다.
다음 재료로 만들 수 있는 요리 하나와 간단한 레시피를 작성하세요.

재료: {ingredients}

출력 형식:
추천 요리:
재료:
조리법:
1.
2.
3.
""")

# 2. LCEL 체인 생성
chain = prompt | llm | StrOutputParser()

# 3. 사용자 입력
ingredients = input("재료 입력: ")

# 4. 실행
result = chain.invoke({"ingredients": ingredients})

print(result)


**추천 요리:** 김치볶음밥·계란 토핑  

**재료:**  
- 김치 1컵 (잘게 썬 것)  
- 밥 2공기 (남은 밥이면 더욱 좋음)  
- 계란 2개  
- 식용유 2큰술  
- 간장 1작은술 (선택)  
- 참기름 ½작은술 (선택)  
- 파·깨소금 약간 (선택)  

**조리법:**  
1. **재료 준비** – 김치는 물기를 살짝 짜서 1cm 정도 크기로 썰고, 계란은 풀어 놓는다.  
2. **김치 볶기** – 팬에 식용유 1큰술을 두르고 중불에서 김치를 2~3분 정도 볶아 향을 낸다. 필요하면 간장을 넣어 간을 맞춘다.  
3. **밥 넣고 볶기** – 김치가 충분히 익으면 밥을 넣고 고루 섞으며 3~4분 정도 볶는다. 마지막에 참기름을 두르고 파·깨소금을 뿌려 마무리한다.  
4. **계란 프라이** – 다른 팬에 식용유 1큰술을 두르고 중불에서 계란을 프라이한다. 반숙 혹은 완숙은 취향에 따라 조절한다.  
5. **완성** – 볶은 김치밥을 그릇에 담고 그 위에 프라이한 계란을 올려 서빙한다.  

**팁**: 매콤함을 원하면 고추장을 ½작은술 정도 추가하거나, 고소함을 더하고 싶다면 다진 마늘과 양파를 함께 볶아도 좋습니다. 즐거운 식사 되세요!


## 1-2 2단계 여행지 체인

In [26]:

# 1. 첫 번째 프롬프트
prompt1 = ChatPromptTemplate.from_messages([
    ("system", "당신은 여행 전문가입니다. 입력된 여행지의 대표 관광 명소 1가지를 추천하세요. 명소 이름만 출력하세요."),
    ("user", "여행지: {destination}")
])

# 2. 두 번째 프롬프트
prompt2 = ChatPromptTemplate.from_messages([
    ("system", "당신은 관광 가이드입니다. 관광 명소의 역사, 특징, 방문 팁을 설명하세요."),
    ("user", """
    여행지: {destination}
    추천 명소: {attraction}

    다음 형식으로 설명하세요.

    명소:
    역사:
    특징:
    방문 팁:
    """)
])

# 3. 각 단계 체인 생성
chain1 = prompt1 | llm | StrOutputParser()
chain2 = prompt2 | llm | StrOutputParser()

# 4. Multi Chain 연결
multi_chain = (
    RunnablePassthrough.assign(attraction=chain1)
    | RunnablePassthrough.assign(detail=chain2)
)

# 5. 실행
destination = input("여행지 입력: ")

result = multi_chain.invoke({"destination": destination})

print("1단계 결과:", result["attraction"])
print("\n2단계 결과:")
print(result["detail"])


1단계 결과: 콜로세움

2단계 결과:
**명소:** 콜로세움 (Colosseo) – 로마 중심부에 위치한 고대 원형극장  

**역사:**  
- 서기 72년~80년 사이에 베스파시아누스 황제와 그의 아들 티투스 황제에 의해 건설된 로마 제국의 대표적인 건축물이다.  
- 원래는 “플라비우스 원형극장”(Amphitheatrum Flavium)이라 불렸으며, 약 5만 명을 수용할 수 있었다.  
- 검투사 경기, 야수 사냥, 해전 재현 등 다양한 공개 오락이 열렸으며, 기독교 박해와 같은 비극적인 사건도 이곳에서 일어났다.  
- 중세 이후에는 돌을 재활용하기 위해 일부가 파괴되었고, 18·19세기에는 보존 운동가들의 노력으로 현재와 같은 형태를 되찾았다.  

**특징:**  
- **구조와 설계:** 4층(지하층 포함)으로 이루어진 복합 구조이며, 아치와 콘크리트 기술이 결합된 로마 건축의 정수다.  
- **외관:** 원형 외벽은 80개의 아치와 3개의 입구(북·동·남)로 구성돼 있어, 일몰 시 특히 장관을 이룬다.  
- **내부:** 지하층(하이포가)에는 동물 우리와 검투사 대기실이 있었으며, 현재는 복원된 전시관과 가이드 투어가 진행된다.  
- **음향:** 관중석에서 중앙 무대로 소리를 전달하는 뛰어난 음향 설계가 돋보이며, 오늘날에도 작은 공연이 열릴 정도다.  

**방문 팁:**  
1. **조기 입장:** 개장 시간인 오전 8시 30분(시즌에 따라 다름) 직전에 도착하면 긴 대기열을 피할 수 있다.  
2. **온라인 티켓 구매:** 공식 웹사이트 또는 인증된 티켓 판매처에서 사전 예약하면 입장권을 바로 스캔해 입장 가능, 현장 구매보다 30~40% 저렴할 수 있다.  
3. **패스 활용:** 로마 48시간 패스(Rome 48‑Hour Pass)나 ‘콜로세움 + 포럼 + 팔라티노’ 복합 티켓을 구매하면 비용 절감과 빠른 입장이 가능하다.  
4. **가이드 투어:** 영어·한국어·스페인어 등 다국어 오디오 가이드를 대여하거나, 현장 가이

## 1-3 Few-Shot 뉴스 키워드 추출기

In [27]:

# 1. Few-Shot 예시
examples = [
    {
        "news": "삼성전자가 AI 가속기를 출시하며 엔비디아에 도전한다.",
        "keywords": "삼성전자, 인공지능, 엔비디아"
    },
    {
        "news": "세계보건기구가 건강 위기 대응을 위한 국제 협력을 강조했다.",
        "keywords": "세계보건기구, 건강위기, 국제협력"
    },
    {
        "news": "현대자동차가 차세대 전기차 배터리 기술을 개발한다.",
        "keywords": "현대자동차, 전기차, 배터리"
    }
]

# 2. 예시 프롬프트
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{news}"),
    ("ai", "키워드: {keywords}")
])

# 3. Few-Shot 프롬프트
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

# 4. 최종 프롬프트
final_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    당신은 뉴스 키워드 추출 전문가입니다.
    뉴스 기사에서 핵심 키워드 3개를 추출하세요.
    출력 형식: 키워드: 단어1, 단어2, 단어3
    추가 설명은 작성하지 마세요.
    """),
    few_shot_prompt,
    ("human", "{input}")
])

# 5. LCEL 체인 생성
chain = final_prompt | llm | StrOutputParser()

# 6. 테스트 뉴스
news = """
제미나이 2.0 플래시는 현재 구글 AI 스튜디오 및
버텍스 AI에서 제미나이 API를 통해 개발자에게
실험 모델로 제공됩니다.
모든 개발자는 멀티모달 입력 및 텍스트 출력을
사용할 수 있으며, 텍스트 음성 변환 및
네이티브 이미지 생성은 일부 파트너들을 대상으로
제공됩니다.
내년 1월에는 더 많은 모델 사이즈와 함께
일반에 공개될 예정입니다.
"""

# 7. 실행
result = chain.invoke({"input": news})

print(result)


키워드: 제미나이 2.0, 멀티모달, 공개


### 다양한 뉴스 분야 추가 테스트

In [28]:
test_news = [
    "삼성전자가 새로운 AI 반도체를 공개했다.",
    "정부가 청년층의 주거 안정을 위한 부동산 정책을 발표했다.",
    "대한민국 축구 국가대표팀이 월드컵 예선에서 승리했다."
]

for i, news in enumerate(test_news, 1):
    result = chain.invoke({"input": news})
    print(f"\n뉴스 {i}")
    print(result)


뉴스 1
키워드: 삼성전자, AI 반도체, 신제품

뉴스 2
키워드: 정부, 청년주거, 부동산정책

뉴스 3
키워드: 대한민국, 축구대표팀, 월드컵예선
